# Classifier Evaluation Quiz Generator

This notebook generates **ground-truth fight events** and **three hypothetical classifier outputs** for students to manually compute:
- TP, FP, TN, FN
- Accuracy, Precision, Recall

Default quiz setup:
- 100 total frames
- 10 fight events
- each event duration = 1 frame

You can regenerate new quizzes by changing the parameters in the final cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def create_ground_truth_events(
    n_frames: int = 100,
    n_events: int = 10,
    event_duration: int = 1,
    seed: int | None = None,
):
    """Create a binary ground-truth timeline with non-overlapping events."""
    if event_duration < 1:
        raise ValueError('event_duration must be >= 1')
    if n_events < 0:
        raise ValueError('n_events must be >= 0')
    if n_events * event_duration > n_frames:
        raise ValueError('n_events * event_duration cannot exceed n_frames')

    rng = np.random.default_rng(seed)
    y_true = np.zeros(n_frames, dtype=int)

    available_starts = np.arange(0, n_frames - event_duration + 1)
    chosen_starts = []

    while len(chosen_starts) < n_events:
        start = int(rng.choice(available_starts))
        overlaps = any(
            (start < s + event_duration) and (s < start + event_duration)
            for s in chosen_starts
        )
        if overlaps:
            continue
        chosen_starts.append(start)

    chosen_starts = sorted(chosen_starts)
    for start in chosen_starts:
        y_true[start : start + event_duration] = 1

    event_frames = np.where(y_true == 1)[0]
    return y_true, event_frames, chosen_starts


def confusion_counts(y_true: np.ndarray, y_pred: np.ndarray):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fp, tn, fn


def compute_metrics(tp: int, fp: int, tn: int, fn: int):
    total = tp + fp + tn + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
    }


def pick_counts_from_targets(
    P: int,
    N: int,
    targets: dict,
    weights: dict | None = None,
    max_predicted_positives: int | None = None,
):
    """
    Find integer (TP, FP) that best matches target metrics.
    P: number of true positive-class frames
    N: number of true negative-class frames
    """
    if weights is None:
        weights = {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0}

    best = None
    for tp in range(P + 1):
        fn = P - tp
        for fp in range(N + 1):
            if max_predicted_positives is not None and (tp + fp) > max_predicted_positives:
                continue

            tn = N - fp
            m = compute_metrics(tp, fp, tn, fn)
            score = 0.0
            for k in ['accuracy', 'precision', 'recall']:
                if k in targets:
                    score += weights.get(k, 1.0) * (m[k] - targets[k]) ** 2
            if best is None or score < best['score']:
                best = {'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn, 'metrics': m, 'score': score}

    return best


def build_prediction_from_counts(
    y_true: np.ndarray,
    tp: int,
    fp: int,
    seed: int | None = None,
):
    rng = np.random.default_rng(seed)
    positive_idx = np.where(y_true == 1)[0]
    negative_idx = np.where(y_true == 0)[0]

    chosen_tp = rng.choice(positive_idx, size=tp, replace=False) if tp > 0 else np.array([], dtype=int)
    chosen_fp = rng.choice(negative_idx, size=fp, replace=False) if fp > 0 else np.array([], dtype=int)

    y_pred = np.zeros_like(y_true, dtype=int)
    y_pred[chosen_tp] = 1
    y_pred[chosen_fp] = 1
    return y_pred

In [ ]:
def plot_timeline(
    rows: dict,
    title: str = 'Ground Truth and Classifier Outputs',
    vertical_gap: float = 0.22,
    guide_color: str = 'gray',
    guide_alpha: float = 0.18,
    guide_linewidth: float = 0.8,
):
    labels = list(rows.keys())
    n_rows = len(labels)

    top_y = 1.0
    y_positions = [top_y - i * vertical_gap for i in range(n_rows)]
    y_min = min(y_positions) - 0.06
    y_max = top_y + 0.06

    fig, ax = plt.subplots(figsize=(14, 2.8))

    ground_truth_label = 'Ground Truth' if 'Ground Truth' in rows else labels[0]
    ground_truth_frames = np.where(rows[ground_truth_label] == 1)[0]
    ax.vlines(
        ground_truth_frames,
        ymin=y_min,
        ymax=y_max,
        colors=guide_color,
        alpha=guide_alpha,
        linewidth=guide_linewidth,
        zorder=0,
    )

    for i, label in enumerate(labels):
        y_level = y_positions[i]
        frames = np.where(rows[label] == 1)[0]
        ax.scatter(frames, np.full_like(frames, y_level, dtype=float), marker='|', s=520, linewidths=2.5)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels)
    ax.set_xlabel('Frame')
    ax.set_title(title)
    ax.grid(False)
    ax.set_xlim(-1, len(next(iter(rows.values()))))
    ax.set_ylim(y_min, y_max)
    plt.tight_layout()
    plt.show()


def generate_classifier_outputs(
    y_true: np.ndarray,
    classifier_profiles: dict,
    seed: int = 0,
    max_predictions_multiplier: float = 2.0,
):
    rng = np.random.default_rng(seed)

    P = int(np.sum(y_true == 1))
    N = int(np.sum(y_true == 0))

    max_predicted_positives = int(np.floor(max_predictions_multiplier * P))

    outputs = {}
    summary_rows = []

    for name, profile in classifier_profiles.items():
        targets = profile['targets']
        weights = profile.get('weights', {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0})

        best = pick_counts_from_targets(
            P=P,
            N=N,
            targets=targets,
            weights=weights,
            max_predicted_positives=max_predicted_positives,
        )

        pred_seed = int(rng.integers(0, 10_000_000))
        y_pred = build_prediction_from_counts(y_true, best['tp'], best['fp'], seed=pred_seed)
        outputs[name] = y_pred

        tp, fp, tn, fn = confusion_counts(y_true, y_pred)
        m = compute_metrics(tp, fp, tn, fn)

        summary_rows.append({
            'classifier': name,
            'profile': profile.get('description', ''),
            'TP': tp,
            'FP': fp,
            'TN': tn,
            'FN': fn,
            'accuracy': m['accuracy'],
            'precision': m['precision'],
            'recall': m['recall'],
            'target_accuracy': targets.get('accuracy', np.nan),
            'target_precision': targets.get('precision', np.nan),
            'target_recall': targets.get('recall', np.nan),
            'max_allowed_predicted_positives': max_predicted_positives,
            'actual_predicted_positives': int(np.sum(y_pred == 1)),
        })

    summary_df = pd.DataFrame(summary_rows)
    return outputs, summary_df


def generate_quiz(
    n_frames: int = 100,
    n_events: int = 10,
    event_duration: int = 1,
    seed: int = 42,
    classifier_profiles: dict | None = None,
    max_predictions_multiplier: float = 2.0,
    guide_color: str = 'gray',
    guide_alpha: float = 0.18,
    guide_linewidth: float = 0.8,
    show_answer_key: bool = False,
):
    if classifier_profiles is None:
        classifier_profiles = {
            'Classifier A': {
                'description': 'High accuracy, low precision',
                'targets': {'accuracy': 0.95, 'precision': 0.7, 'recall': 0.7},
                'weights': {'accuracy': 2, 'precision': 1.3, 'recall': 1.0},
            },
            'Classifier B': {
                'description': 'High precision, low recall',
                'targets': {'accuracy': 0.9, 'precision': 1.00, 'recall': 0.30},
                'weights': {'accuracy': 2.0, 'precision': 1.4, 'recall': 0.4},
            },
            'Classifier C': {
                'description': 'High recall, low precision',
                'targets': {'accuracy': 0.83, 'precision': 0.35, 'recall': 0.95},
                'weights': {'accuracy': 1.0, 'precision': 1.2, 'recall': 1.4},
            },
        }

    y_true, event_frames, event_starts = create_ground_truth_events(
        n_frames=n_frames,
        n_events=n_events,
        event_duration=event_duration,
        seed=seed,
    )

    outputs, summary_df = generate_classifier_outputs(
        y_true=y_true,
        classifier_profiles=classifier_profiles,
        seed=seed + 1,
        max_predictions_multiplier=max_predictions_multiplier,
    )

    rows = {'Ground Truth': y_true}
    rows.update(outputs)
    plot_timeline(
        rows,
        title='Fight Events Over Timeline',
        vertical_gap=0.22,
        guide_color=guide_color,
        guide_alpha=guide_alpha,
        guide_linewidth=guide_linewidth,
    )

    print(f'Ground-truth event frames ({len(event_frames)} total):')
    print(event_frames.tolist())

    print('\nClassifier positive frames (predicted fights):')
    for name, y_pred in outputs.items():
        pred_frames = np.where(y_pred == 1)[0].tolist()
        print(f'- {name}: {pred_frames} (count={len(pred_frames)}, max_allowed={int(max_predictions_multiplier * len(event_frames))})')

    worksheet_df = pd.DataFrame({
        'Classifier': list(outputs.keys()),
        'TP': [''] * len(outputs),
        'FP': [''] * len(outputs),
        'TN': [''] * len(outputs),
        'FN': [''] * len(outputs),
        'Accuracy': [''] * len(outputs),
        'Precision': [''] * len(outputs),
        'Recall': [''] * len(outputs),
    })

    print('\nStudent worksheet (fill these manually):')
    display(worksheet_df)

    if show_answer_key:
        answer_cols = [
            'classifier', 'profile',
            'TP', 'FP', 'TN', 'FN',
            'accuracy', 'precision', 'recall',
            'target_accuracy', 'target_precision', 'target_recall',
            'actual_predicted_positives', 'max_allowed_predicted_positives'
        ]
        print('\nAnswer key (computed):')
        display(summary_df[answer_cols].copy())

    return {
        'y_true': y_true,
        'outputs': outputs,
        'summary': summary_df,
        'event_frames': event_frames,
        'event_starts': event_starts,
    }

## Run default quiz
This matches your requested setup: 100 frames, 10 one-frame events.

In [ ]:
quiz = generate_quiz(
    n_frames=100,
    n_events=10,
    event_duration=1,
    seed=7,
    show_answer_key=True,  # Keep False for students
    guide_color='k',
    guide_alpha=0.2,
    guide_linewidth=1.0,
)

## Optional: regenerate with custom settings
Edit targets to create new quizzes with different classifier profiles.

In [ ]:
custom_profiles = {
    'Classifier A': {
        'description': 'High accuracy, low precision',
        'targets': {'accuracy': 0.90, 'precision': 0.30, 'recall': 0.75},
    },
    'Classifier B': {
        'description': 'High precision, low recall',
        'targets': {'accuracy': 0.93, 'precision': 0.92, 'recall': 0.25},
    },
    'Classifier C': {
        'description': 'High recall, low precision',
        'targets': {'accuracy': 0.82, 'precision': 0.33, 'recall': 0.95},
    },
}

# Example run for instructor preview:
_ = generate_quiz(
    n_frames=100,
    n_events=10,
    event_duration=1,
    seed=7,
    classifier_profiles=custom_profiles,
    show_answer_key=True,
)